# Extended Pipeline: Using Full Work History

This notebook extends the base model by incorporating information from **all positions** in a person's CV, not just the current (ACTIVE) one.

**Key assumptions:**
1. Seniority typically increases over time
2. People rarely change departments
3. Organization name provides context (NGO, University, etc.)
4. Career length correlates with seniority
5. Position count indicates career stage

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sentence_transformers import SentenceTransformer

## 1. Load Data

In [ ]:
# Load training data
df_dept_train = pd.read_csv('./data/department-v2.csv')
df_sen_train = pd.read_csv('./data/seniority-v2.csv')

print(f"Department training: {len(df_dept_train)}")
print(f"Seniority training: {len(df_sen_train)}")

In [ ]:
# Load full CV data (all positions, not just ACTIVE)
def load_linkedin_full(filepath):
    """Load LinkedIn data preserving person grouping."""
    data = json.loads(Path(filepath).read_text(encoding='utf-8'))
    return data  # List of persons, each person is a list of positions

test_persons = load_linkedin_full('./data/linkedin-cvs-annotated.json')
predict_persons = load_linkedin_full('./data/linkedin-cvs-not-annotated.json')

print(f"Test: {len(test_persons)} persons")
print(f"Predict: {len(predict_persons)} persons")

# Count total and active positions
test_total = sum(len(p) for p in test_persons)
test_active = sum(1 for p in test_persons for j in p if j.get('status') == 'ACTIVE')
print(f"\nTest positions: {test_total} total, {test_active} ACTIVE")

## 2. Data Augmentation

In [ ]:
# Synthetic examples for underrepresented classes
other_examples = [
    'CEO', 'CFO', 'COO', 'CTO', 'CMO', 'CHRO', 'CIO',
    'Chief Executive Officer', 'Chief Financial Officer', 'Chief Operating Officer',
    'Managing Director', 'General Manager', 'President', 'Vice President',
    'Geschäftsführer', 'Geschäftsführerin', 'Geschäftsführender Gesellschafter',
    'Vorstand', 'Vorstandsvorsitzender', 'Vorstandsmitglied',
    'Founder', 'Co-Founder', 'Owner', 'Inhaber', 'Partner', 'Gesellschafter',
    'Board Member', 'Board Director', 'Executive Director',
    'Financial Controller', 'Accountant', 'Buchhalter', 'Financial Analyst',
    'Treasurer', 'Controller', 'Auditor', 'Investment Analyst', 'Portfolio Manager',
    'Lawyer', 'Attorney', 'Legal Counsel', 'Rechtsanwalt', 'Rechtsanwältin',
    'Jurist', 'Paralegal', 'Legal Advisor', 'General Counsel',
    'Professor', 'Associate Professor', 'Assistant Professor',
    'Researcher', 'Research Assistant', 'Scientist', 'Research Scientist',
    'PhD Student', 'Postdoc', 'Postdoctoral Researcher', 'Dozent',
    'Operations Manager', 'Supply Chain Manager', 'Logistics Manager',
    'Betriebsleiter', 'Facility Manager', 'Plant Manager',
    'Intern', 'Trainee', 'Praktikant', 'Praktikantin', 'Werkstudent', 'Werkstudentin',
    'Apprentice', 'Auszubildender', 'Auszubildende', 'Working Student',
    'Product Manager', 'Product Owner', 'Produktmanager',
    'Doctor', 'Physician', 'Arzt', 'Ärztin', 'Nurse', 'Pharmacist',
    'Consultant', 'Freelancer', 'Contractor', 'Self-employed', 'Selbstständig',
]

professional_examples = [
    'Consultant', 'Berater', 'Specialist', 'Spezialist', 'Analyst',
    'Engineer', 'Developer', 'Coordinator', 'Administrator',
    'Software Engineer', 'Software Developer', 'Softwareentwickler',
    'Data Analyst', 'Business Analyst', 'Systems Analyst',
    'Web Developer', 'Frontend Developer', 'Backend Developer',
    'DevOps Engineer', 'QA Engineer', 'Test Engineer',
    'Account Manager', 'Project Manager', 'Projektmanager',
    'Sales Representative', 'Marketing Specialist', 'HR Specialist',
    'Recruiter', 'Financial Analyst', 'Business Developer',
    'Sachbearbeiter', 'Sachbearbeiterin', 'Referent', 'Referentin',
    'Mitarbeiter', 'Mitarbeiterin', 'Fachkraft',
    'Researcher', 'Scientist', 'Professor', 'Lawyer', 'Rechtsanwalt',
    'Architect', 'Designer', 'Writer', 'Editor', 'Journalist',
    'Nurse', 'Pharmacist', 'Technician', 'Electrician', 'Mechanic',
]

# Augment training data
df_dept_aug = pd.concat([
    df_dept_train,
    pd.DataFrame({'text': other_examples, 'label': 'Other'})
], ignore_index=True)

df_sen_aug = pd.concat([
    df_sen_train,
    pd.DataFrame({'text': professional_examples, 'label': 'Professional'})
], ignore_index=True)

print(f"Augmented department: {len(df_dept_aug)}")
print(f"Augmented seniority: {len(df_sen_aug)}")

## 3. Train Base Classifiers

In [ ]:
# Load encoder
print("Loading embedding model...")
encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Done!")

In [ ]:
# Create training embeddings
print("Creating training embeddings...")
X_train_dept = encoder.encode(df_dept_aug['text'].tolist(), show_progress_bar=True, batch_size=64)
y_train_dept = df_dept_aug['label'].values

X_train_sen = encoder.encode(df_sen_aug['text'].tolist(), show_progress_bar=True, batch_size=64)
y_train_sen = df_sen_aug['label'].values

print(f"Train dept: {X_train_dept.shape}")
print(f"Train sen: {X_train_sen.shape}")

In [ ]:
# Train classifiers
print("Training classifiers...")

clf_dept = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)
clf_dept.fit(X_train_dept, y_train_dept)

clf_sen = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)
clf_sen.fit(X_train_sen, y_train_sen)

print("Done!")

## 4. Define History-Based Adjustments

In [ ]:
# Seniority level ordering (for trajectory analysis)
SENIORITY_ORDER = ['Junior', 'Professional', 'Senior', 'Lead', 'Management', 'Director']
SENIORITY_LEVEL = {s: i for i, s in enumerate(SENIORITY_ORDER)}

def sen_to_level(sen):
    """Convert seniority label to numeric level."""
    return SENIORITY_LEVEL.get(sen, 2)  # Default to Professional level

def level_to_sen(level):
    """Convert numeric level to seniority label."""
    level = max(0, min(level, len(SENIORITY_ORDER) - 1))
    return SENIORITY_ORDER[level]

In [ ]:
# Organization patterns that indicate specific departments
ORG_PATTERNS = {
    # Pattern -> (likely department, seniority hints)
    r'\b(ngo|foundation|stiftung|charity|nonprofit|non-profit|gemeinnützig)\b': 
        {'dept': 'Other', 'notes': 'NGO/nonprofit'},
    r'\b(university|universität|college|hochschule|institute|institut|school|schule|akademie)\b': 
        {'dept': 'Other', 'notes': 'Academia'},
    r'\b(hospital|klinik|clinic|medical|krankenhaus|health|gesundheit)\b': 
        {'dept': 'Other', 'notes': 'Healthcare'},
    r'\b(law firm|kanzlei|rechtsanwalt|legal|attorney)\b': 
        {'dept': 'Other', 'notes': 'Legal'},
    r'\b(bank|capital|investment|finance|finanz|asset management|versicherung|insurance)\b': 
        {'dept': 'Finance', 'notes': 'Finance industry'},
    r'\b(consulting|beratung|advisory|berater)\b': 
        {'dept': 'Consulting', 'notes': 'Consulting'},
}

def get_org_hints(org_name):
    """Extract department hints from organization name."""
    if not org_name:
        return None
    org_lower = str(org_name).lower()
    
    for pattern, hints in ORG_PATTERNS.items():
        if re.search(pattern, org_lower):
            return hints
    return None

In [ ]:
def parse_date(date_dict):
    """Parse date dictionary to datetime."""
    if not date_dict:
        return None
    try:
        year = date_dict.get('year', 2000)
        month = date_dict.get('month', 1)
        day = date_dict.get('day', 1)
        return datetime(year, month, day)
    except:
        return None

def get_career_length_years(positions):
    """Calculate career length from earliest position."""
    dates = []
    for p in positions:
        d = parse_date(p.get('started_on'))
        if d:
            dates.append(d)
    
    if not dates:
        return 0
    
    earliest = min(dates)
    return (datetime.now() - earliest).days / 365

def get_position_order(positions):
    """Order positions chronologically (oldest first)."""
    def get_start_date(p):
        d = parse_date(p.get('started_on'))
        return d if d else datetime.min
    
    return sorted(range(len(positions)), key=lambda i: get_start_date(positions[i]))

In [ ]:
def predict_with_history(person_positions, clf_dept, clf_sen, encoder, 
                         dept_threshold=0.5, sen_threshold=0.5):
    """
    Predict department and seniority using full work history.
    
    Returns predictions for the ACTIVE position, adjusted based on:
    1. Career trajectory (seniority should increase)
    2. Department consistency (rarely changes)
    3. Organization context
    4. Career length
    """
    
    # Find ACTIVE position
    active_idx = None
    for i, p in enumerate(person_positions):
        if p.get('status') == 'ACTIVE':
            active_idx = i
            break
    
    if active_idx is None:
        return 'Other', 'Professional'  # Default if no ACTIVE found
    
    # Get all position titles and embeddings
    all_titles = [p.get('position', '') for p in person_positions]
    all_embeddings = encoder.encode(all_titles)
    
    # Base predictions for all positions
    dept_preds = clf_dept.predict(all_embeddings)
    dept_probas = clf_dept.predict_proba(all_embeddings)
    
    sen_preds = clf_sen.predict(all_embeddings)
    sen_probas = clf_sen.predict_proba(all_embeddings)
    
    # Current position predictions
    active_dept = dept_preds[active_idx]
    active_dept_conf = dept_probas[active_idx].max()
    
    active_sen = sen_preds[active_idx]
    active_sen_conf = sen_probas[active_idx].max()
    
    # === ADJUSTMENT 1: Department Consistency ===
    # If career shows consistent department, trust that over uncertain current prediction
    dept_counts = Counter(dept_preds)
    most_common_dept, most_common_count = dept_counts.most_common(1)[0]
    dept_consistency = most_common_count / len(dept_preds)
    
    if active_dept != most_common_dept and active_dept_conf < 0.6 and dept_consistency > 0.6:
        # Current prediction differs from career trend AND is uncertain AND trend is strong
        active_dept = most_common_dept
    
    # === ADJUSTMENT 2: Seniority Trajectory ===
    # Seniority should generally increase over time
    position_order = get_position_order(person_positions)
    
    # Get seniority levels in chronological order
    sen_levels = [sen_to_level(sen_preds[i]) for i in position_order]
    
    # Find max seniority before current position
    active_order_idx = position_order.index(active_idx)
    prev_levels = sen_levels[:active_order_idx]
    max_prev_level = max(prev_levels) if prev_levels else 0
    
    active_sen_level = sen_to_level(active_sen)
    
    # If current seniority is much lower than previous max, adjust
    if active_sen_level < max_prev_level - 1 and active_sen_conf < 0.7:
        # Unlikely regression - set to at most 1 level below max
        active_sen = level_to_sen(max(active_sen_level, max_prev_level - 1))
    
    # === ADJUSTMENT 3: Organization Context ===
    active_org = person_positions[active_idx].get('organization', '')
    org_hints = get_org_hints(active_org)
    
    if org_hints and active_dept_conf < 0.6:
        # Low confidence + organization hint -> use the hint
        if 'dept' in org_hints:
            active_dept = org_hints['dept']
    
    # === ADJUSTMENT 4: Career Length ===
    career_years = get_career_length_years(person_positions)
    
    if career_years > 15 and active_sen in ['Junior']:
        # 15+ years of experience unlikely to be Junior
        active_sen = 'Professional'
    elif career_years > 20 and active_sen in ['Junior', 'Professional']:
        # 20+ years -> at least Senior
        active_sen = 'Senior'
    
    # === ADJUSTMENT 5: Position Count ===
    num_positions = len(person_positions)
    
    if num_positions >= 8 and active_sen in ['Junior']:
        # Many positions -> not a Junior
        active_sen = 'Professional'
    
    # === Apply confidence threshold (fallback to underrepresented class) ===
    if active_dept_conf < dept_threshold:
        active_dept = 'Other'
    
    if active_sen_conf < sen_threshold:
        active_sen = 'Professional'
    
    return active_dept, active_sen

## 5. Evaluate: Base Model vs Extended Model

In [ ]:
def predict_base_model(position, clf_dept, clf_sen, encoder, dept_threshold=0.5, sen_threshold=0.5):
    """Base model: predict using only the current position."""
    emb = encoder.encode([position])
    
    dept_pred = clf_dept.predict(emb)[0]
    dept_conf = clf_dept.predict_proba(emb).max()
    
    sen_pred = clf_sen.predict(emb)[0]
    sen_conf = clf_sen.predict_proba(emb).max()
    
    if dept_conf < dept_threshold:
        dept_pred = 'Other'
    if sen_conf < sen_threshold:
        sen_pred = 'Professional'
    
    return dept_pred, sen_pred

In [ ]:
# Evaluate both models
print("Evaluating models...")

results_base = {'dept_pred': [], 'sen_pred': [], 'dept_true': [], 'sen_true': []}
results_ext = {'dept_pred': [], 'sen_pred': [], 'dept_true': [], 'sen_true': []}

for person_idx, person in enumerate(test_persons):
    # Find ACTIVE position
    active_pos = None
    for job in person:
        if job.get('status') == 'ACTIVE':
            active_pos = job
            break
    
    if active_pos is None:
        continue
    
    # Ground truth
    true_dept = active_pos.get('department')
    true_sen = active_pos.get('seniority')
    
    if not true_dept or not true_sen:
        continue
    
    # Base model prediction (only current position)
    base_dept, base_sen = predict_base_model(
        active_pos.get('position', ''), clf_dept, clf_sen, encoder
    )
    
    # Extended model prediction (full history)
    ext_dept, ext_sen = predict_with_history(
        person, clf_dept, clf_sen, encoder
    )
    
    results_base['dept_pred'].append(base_dept)
    results_base['sen_pred'].append(base_sen)
    results_base['dept_true'].append(true_dept)
    results_base['sen_true'].append(true_sen)
    
    results_ext['dept_pred'].append(ext_dept)
    results_ext['sen_pred'].append(ext_sen)
    results_ext['dept_true'].append(true_dept)
    results_ext['sen_true'].append(true_sen)

print(f"Evaluated {len(results_base['dept_pred'])} ACTIVE positions")

In [ ]:
print("=" * 70)
print("COMPARISON: BASE MODEL vs EXTENDED MODEL")
print("=" * 70)

print("\n" + "-" * 70)
print("DEPARTMENT CLASSIFICATION")
print("-" * 70)

base_dept_acc = accuracy_score(results_base['dept_true'], results_base['dept_pred'])
ext_dept_acc = accuracy_score(results_ext['dept_true'], results_ext['dept_pred'])

base_dept_f1 = f1_score(results_base['dept_true'], results_base['dept_pred'], average='macro')
ext_dept_f1 = f1_score(results_ext['dept_true'], results_ext['dept_pred'], average='macro')

print(f"{'Model':<30} {'Accuracy':>12} {'Macro F1':>12}")
print("-" * 55)
print(f"{'Base (current position only)':<30} {base_dept_acc:>12.3f} {base_dept_f1:>12.3f}")
print(f"{'Extended (full history)':<30} {ext_dept_acc:>12.3f} {ext_dept_f1:>12.3f}")
print(f"{'Improvement':<30} {ext_dept_acc - base_dept_acc:>+12.3f} {ext_dept_f1 - base_dept_f1:>+12.3f}")

In [ ]:
print("\n" + "-" * 70)
print("SENIORITY CLASSIFICATION")
print("-" * 70)

base_sen_acc = accuracy_score(results_base['sen_true'], results_base['sen_pred'])
ext_sen_acc = accuracy_score(results_ext['sen_true'], results_ext['sen_pred'])

base_sen_f1 = f1_score(results_base['sen_true'], results_base['sen_pred'], average='macro')
ext_sen_f1 = f1_score(results_ext['sen_true'], results_ext['sen_pred'], average='macro')

print(f"{'Model':<30} {'Accuracy':>12} {'Macro F1':>12}")
print("-" * 55)
print(f"{'Base (current position only)':<30} {base_sen_acc:>12.3f} {base_sen_f1:>12.3f}")
print(f"{'Extended (full history)':<30} {ext_sen_acc:>12.3f} {ext_sen_f1:>12.3f}")
print(f"{'Improvement':<30} {ext_sen_acc - base_sen_acc:>+12.3f} {ext_sen_f1 - base_sen_f1:>+12.3f}")

In [ ]:
print("\n" + "=" * 70)
print("DETAILED CLASSIFICATION REPORTS")
print("=" * 70)

print("\nEXTENDED MODEL - DEPARTMENT:")
print(classification_report(results_ext['dept_true'], results_ext['dept_pred'], digits=3))

print("\nEXTENDED MODEL - SENIORITY:")
print(classification_report(results_ext['sen_true'], results_ext['sen_pred'], digits=3))

## 6. Analyze Where History Helps

In [ ]:
# Find cases where extended model corrected base model
corrections_dept = []
corrections_sen = []

for i in range(len(results_base['dept_true'])):
    true_dept = results_base['dept_true'][i]
    base_dept = results_base['dept_pred'][i]
    ext_dept = results_ext['dept_pred'][i]
    
    # Base wrong, extended correct
    if base_dept != true_dept and ext_dept == true_dept:
        corrections_dept.append({
            'true': true_dept,
            'base': base_dept,
            'ext': ext_dept
        })
    
    true_sen = results_base['sen_true'][i]
    base_sen = results_base['sen_pred'][i]
    ext_sen = results_ext['sen_pred'][i]
    
    if base_sen != true_sen and ext_sen == true_sen:
        corrections_sen.append({
            'true': true_sen,
            'base': base_sen,
            'ext': ext_sen
        })

print(f"Department corrections (base wrong → ext correct): {len(corrections_dept)}")
print(f"Seniority corrections (base wrong → ext correct): {len(corrections_sen)}")

In [ ]:
# What types of corrections were made?
if corrections_dept:
    print("\nDepartment correction patterns:")
    correction_patterns = Counter([(c['base'], c['true']) for c in corrections_dept])
    for (base, true), count in correction_patterns.most_common(10):
        print(f"  {base} → {true}: {count}")

In [ ]:
if corrections_sen:
    print("\nSeniority correction patterns:")
    correction_patterns = Counter([(c['base'], c['true']) for c in corrections_sen])
    for (base, true), count in correction_patterns.most_common(10):
        print(f"  {base} → {true}: {count}")

## 7. Apply to Unlabeled Data

In [ ]:
# Predict on unlabeled data using extended model
print("Predicting on unlabeled data...")

predictions = []

for person_idx, person in enumerate(predict_persons):
    # Find ACTIVE position
    active_pos = None
    for job in person:
        if job.get('status') == 'ACTIVE':
            active_pos = job
            break
    
    if active_pos is None:
        continue
    
    # Extended model prediction
    dept_pred, sen_pred = predict_with_history(
        person, clf_dept, clf_sen, encoder
    )
    
    predictions.append({
        'person_id': person_idx,
        'position': active_pos.get('position'),
        'organization': active_pos.get('organization'),
        'num_positions': len(person),
        'department_pred': dept_pred,
        'seniority_pred': sen_pred,
    })

df_predictions = pd.DataFrame(predictions)
print(f"\nPredicted {len(df_predictions)} ACTIVE positions")

In [ ]:
print("\nPredicted distributions:")
print("\nDepartment:")
print(df_predictions['department_pred'].value_counts())

print("\nSeniority:")
print(df_predictions['seniority_pred'].value_counts())

In [ ]:
# Sample predictions
print("\nSample predictions:")
print(df_predictions.sample(20).to_string())

## 8. Save Results

In [ ]:
import joblib

# Save model and predictions
joblib.dump({
    'dept_classifier': clf_dept,
    'sen_classifier': clf_sen,
    'encoder_name': 'paraphrase-multilingual-MiniLM-L12-v2',
    'seniority_order': SENIORITY_ORDER,
    'org_patterns': ORG_PATTERNS,
}, './data/extended_pipeline_model.joblib')

df_predictions.to_csv('./data/predictions_extended.csv', index=False)

print("Model saved to: ./data/extended_pipeline_model.joblib")
print("Predictions saved to: ./data/predictions_extended.csv")

## 9. Summary

In [ ]:
print("=" * 70)
print("EXTENDED PIPELINE SUMMARY")
print("=" * 70)

print("\n📊 RESULTS:")
print(f"   Base Model Dept Accuracy:     {base_dept_acc:.1%}")
print(f"   Extended Model Dept Accuracy: {ext_dept_acc:.1%} ({ext_dept_acc - base_dept_acc:+.1%})")
print(f"   Base Model Sen Accuracy:      {base_sen_acc:.1%}")
print(f"   Extended Model Sen Accuracy:  {ext_sen_acc:.1%} ({ext_sen_acc - base_sen_acc:+.1%})")

print("\n🔧 ADJUSTMENTS APPLIED:")
print("   1. Department consistency (career trend)")
print("   2. Seniority trajectory (must increase over time)")
print("   3. Organization context (NGO, University, etc.)")
print("   4. Career length heuristics")
print("   5. Position count heuristics")

print("\n📈 IMPROVEMENT ANALYSIS:")
print(f"   Department corrections: {len(corrections_dept)}")
print(f"   Seniority corrections: {len(corrections_sen)}")